这是一个非常前沿且具有挑战性的需求。

**Semantic Kernel (SK)** 和 **MCP (Model Context Protocol)** 是两个独立的生态。
*   **SK** 使用 `Plugin` (类 + 装饰器) 来封装工具。
*   **MCP** 使用 `Client-Server` 协议来暴露工具。

要让 SK 调用 MCP 工具，我们需要写一个 **“适配器 (Adapter)”**。

核心思路是：
1.  **启动 MCP Client**：连接到一个正在运行的 MCP Server。
2.  **获取工具列表**：通过 MCP 协议拿到 Server 上有哪些工具（比如 `get_weather`）。
3.  **构建 SK Plugin**：创建一个通用的 SK Plugin，充当“路由器”。
4.  **注入 Prompt**：把 MCP 的工具描述塞给 LLM，告诉它：“你可以通过调用我的路由函数来使用这些 MCP 工具”。

下面是完整的代码实现。我们将分为两部分：**模拟 MCP 服务器** 和 **SK 客户端适配**。

---

### 第一步：准备环境

你需要安装 MCP 的官方 Python SDK：

```bash
pip install mcp semantic-kernel
```

### 第二步：创建一个简单的 MCP 服务器 (`myserver.py`)

为了演示，我们先写一个最简单的 MCP 服务器，它提供一个“计算器”和一个“天气查询”工具。
请在项目根目录创建 `myserver.py`：

```python
# myserver.py
from mcp.server.fastmcp import FastMCP

# 初始化一个名为 "DemoServer" 的 MCP 服务器
mcp = FastMCP("DemoServer")

@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

@mcp.tool()
def get_weather(city: str) -> str:
    """Get the weather of a specific city."""
    if "Beijing" in city:
        return "Sunny, 25°C"
    elif "Shanghai" in city:
        return "Rainy, 20°C"
    else:
        return "Unknown weather"

if __name__ == "__main__":
    # 使用 stdio 模式运行，这是 MCP 的标准通信方式
    mcp.run(transport='stdio')
```

---

### 第三步：编写 SK 适配器并调用 (`main_mcp.py`)

这是核心部分。我们将编写一个 `McpBridgePlugin`，它负责把 SK 的指令转发给 MCP Server。

```python
# main_mcp.py
import asyncio
import os
import json
import dotenv
import sys

# Semantic Kernel 导入
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion, OpenAIChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function

# MCP 导入
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

dotenv.load_dotenv()

# ==============================================================================
# 核心：定义 MCP 桥接插件 (Adapter)
# ==============================================================================
class McpBridgePlugin:
    """
    这个插件充当 Semantic Kernel 和 MCP Server 之间的桥梁。
    """
    def __init__(self, session: ClientSession):
        self.session = session

    @kernel_function(description="调用外部 MCP 工具执行任务", name="call_mcp_tool")
    async def call_mcp_tool(self, tool_name: str, arguments_json: str) -> str:
        """
        通用路由函数：LLM 会把工具名和参数传给这个函数，然后转发给 MCP。
        """
        print(f"    [🌉 MCP Bridge] 正在转发请求 -> 工具: {tool_name}, 参数: {arguments_json}")

        try:
            # 解析参数 (LLM 有时给的是 JSON 字符串)
            args = json.loads(arguments_json) if isinstance(arguments_json, str) else arguments_json

            # --- 真正的 MCP 调用发生在这里 ---
            result = await self.session.call_tool(tool_name, arguments=args)

            # MCP 返回的是一个对象，我们需要提取文本内容
            output_text = result.content[0].text
            print(f"    [✅ MCP 响应] {output_text}")
            return output_text

        except Exception as e:
            error_msg = f"调用 MCP 工具失败: {str(e)}"
            print(f"    [❌ Error] {error_msg}")
            return error_msg

# ==============================================================================
# 主逻辑
# ==============================================================================
async def main():
    # 1. 配置 MCP 服务器连接参数 (连接到我们刚才写的 myserver.py)
    server_params = StdioServerParameters(
        command="python", # 使用 python 运行
        args=["myserver.py"], # 脚本路径
        env=None
    )

    print(">>> 正在连接 MCP 服务器...")

    # 2. 建立 MCP 连接上下文
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # 初始化 MCP 会话
            await session.initialize()

            # --- 关键步骤：动态获取 MCP 工具列表 ---
            # 我们需要知道服务器上有哪些工具，以便告诉 LLM
            tools_response = await session.list_tools()
            mcp_tools = tools_response.tools

            # 构建工具描述字符串 (System Prompt)
            # 因为我们用的是“路由模式”，LLM 需要知道有哪些工具名和参数结构可用
            tools_desc = []
            for t in mcp_tools:
                tools_desc.append(f"- 工具名: {t.name}\n  描述: {t.description}\n  参数Schema: {t.inputSchema}")

            system_prompt_tools = "\n".join(tools_desc)
            print(f">>> 发现 {len(mcp_tools)} 个 MCP 工具:\n{system_prompt_tools}")

            # --------------------------------------------------------
            # 开始 Semantic Kernel 流程
            # --------------------------------------------------------
            kernel = Kernel()

            # 配置 AI 服务
            kernel.add_service(
                OpenAIChatCompletion(
                    service_id="default",
                    ai_model_id="gpt-4o-mini",
                    api_key=os.getenv("OPENAI_API_KEY"),
                )
            )

            # 3. 注册我们的桥接插件 (注入 session)
            kernel.add_plugin(McpBridgePlugin(session), plugin_name="McpTools")

            # 4. 构建 System Prompt
            # 我们必须明确告诉 LLM：如果想用这些工具，请调用 McpTools-call_mcp_tool
            system_prompt = f"""
            你是一个拥有 MCP 工具扩展能力的助手。
            你可以使用的外部工具如下：
            {system_prompt_tools}

            **重要指令**：
            如果你需要使用上述任何工具，请务必调用插件 `McpTools` 中的 `call_mcp_tool` 函数。
            - `tool_name` 填入上面列表中的工具名。
            - `arguments_json` 填入符合 Schema 的 JSON 字符串。
            """

            # 5. 配置自动调用
            settings = OpenAIChatPromptExecutionSettings(
                service_id="default",
                function_choice_behavior=FunctionChoiceBehavior.Auto()
            )

            chat_service = kernel.get_service("default")
            history = ChatHistory()
            history.add_system_message(system_prompt)

            # --- 测试任务 ---
            user_input = "你好，请帮我查询一下 Shanghai 的天气，顺便算一下 123 加 456 等于多少？"
            print(f"\n[用户]: {user_input}")
            history.add_user_message(user_input)

            # 执行
            result = await chat_service.get_chat_message_contents(
                chat_history=history,
                settings=settings,
                kernel=kernel
            )

            print("\n" + "="*50)
            print(f"[AI 回答]: {result[0]}")
            print("="*50)

if __name__ == "__main__":
    asyncio.run(main())
```

### 代码原理解析

这套方案使用了 **"Router Plugin" (路由插件模式)**。

1.  **为什么不直接把 MCP 工具注册成 SK Functions?**
    *   SK 的 Function 需要明确的 Python 函数签名 (`def func(a: int)`) 来生成 Schema。
    *   MCP 的工具是动态的 JSON Schema。
    *   虽然可以通过元编程动态生成 Python 函数，但代码极其复杂且不稳定。

2.  **路由模式的优势 (`McpBridgePlugin`)**：
    *   我们只定义**一个**通用的 SK Function: `call_mcp_tool(tool_name, args)`。
    *   我们将 MCP Server 提供的工具列表（Schema）格式化成文本，放在 **System Prompt** 里。
    *   **流程**：
        1.  LLM 看到 System Prompt，知道有 `get_weather` 这个能力。
        2.  LLM 决定调用 `call_mcp_tool`，参数填 `tool_name="get_weather", args="{city: Shanghai}"`。
        3.  Python 代码收到请求，通过 `session.call_tool` 转发给 MCP Server。
        4.  MCP Server 执行并返回结果。

### 运行效果

1.  运行 `python main_mcp.py`。
2.  程序会自动启动 `myserver.py` 子进程。
3.  SK 连接到 MCP，获取到 `add_numbers` 和 `get_weather`。
4.  LLM 思考后，会连续调用两次 `call_mcp_tool`（一次查天气，一次算加法）。
5.  最终输出结果：
    ```text
    [AI 回答]: 上海的天气是 Rainy, 20°C，另外 123 加上 456 等于 579。
    ```

### 扩展应用
这个 `McpBridgePlugin` 是通用的。
你可以把 `server_params` 里的 `command` 改成任何标准的 MCP Server：
*   **SQLite MCP**: 让 LLM 操作本地数据库文件。
*   **Filesystem MCP**: 让 LLM 读写本地文件。
*   **GitHub MCP**: 让 LLM 操作你的仓库。

只要那个 Server 符合 MCP 标准，SK 就能通过这个桥接器控制它，而无需修改 Python 代码。